# IL 轨迹预测训练示例

演示 `il` 模块的数据加载与训练流程：
1. 构建 Hydra 风格的 DictConfig
2. 使用 create_dataloaders 创建 DataLoader
3. 检查数据集和 batch 结构
4. 构建模型并执行一次前向传播
5. 调用 train 进行完整训练

In [ ]:
import torch
from pathlib import Path
from omegaconf import OmegaConf
from hydra import compose, initialize_config_dir
from il.data.visualization.trajectory_dataset_vis import TrajectoryDatasetVisualizer
from il.data.visualization.diffusion_process_vis import DiffusionProcessVisualizer
from trainflow.hydra_build import run_fit, run_validate, run_test, run_predict, instantiate_trainflow

import os

def find_project_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "pyproject.toml").exists():
            return cand
    raise RuntimeError("找不到项目根目录（未发现 pyproject.toml）")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
print("cwd:", Path.cwd())

%cd {PROJECT_ROOT}

## 1. 构建配置

il 模块使用 Hydra DictConfig，包含 `data`、`model`、`train`、`loss`、`log` 五个子配置。

`data.data_dir` 应指向包含 TrainingSample pkl 文件的目录。

In [ ]:
from datetime import datetime
from hydra.core.hydra_config import HydraConfig

OmegaConf.register_new_resolver(
    "now", lambda fmt: datetime.now().strftime(fmt), replace=True
)

conf_dir = (Path.cwd() / "src/il/conf").resolve()
if not conf_dir.is_dir():
    raise FileNotFoundError(f"未找到 Hydra 配置目录: {conf_dir}")

with initialize_config_dir(version_base=None, config_dir=str(conf_dir)):
    il_cfg = compose(
        config_name="train_config",
        overrides=[
            "training@_global_=train_traj_diffusion",
            # "run_mode=validate"
            "resume_checkpoint=log/il_train_traj_diffusion_train_20260516_151351/checkpoint/final.ckpt",
            "trainflow.data.train_ratio=0.9",
            "trainflow.data.val_ratio=0.09",
            "trainflow.data.batch_size=4",
            # "trainflow.trainer.max_epochs=10",
            "trainflow.model.predictor.df_type=ddim_rand", # ddpm_eps, ddpm_x0, ddim, ddim_rand
            "hydra.job.num=0",
        ],
        return_hydra_config=True,
    )

HydraConfig.instance().set_config(il_cfg)

print(il_cfg)
# print(OmegaConf.to_yaml(il_cfg, resolve=False))

In [ ]:
trainer, model, datamodule = instantiate_trainflow(il_cfg)

datamodule.setup()
train_loader = datamodule.train_dataloader()
val_loader = datamodule.val_dataloader()
test_loader = datamodule.test_dataloader()
print("train/val/test batches:", len(train_loader), len(val_loader), len(test_loader))

batch = next(iter(test_loader))
future = model.normalizer.normalize_future(batch["future"])
x_noisy, eps, timesteps = model.predictor.order_diffusion_forward(future, 40)

samples = batch
samples.update({"x_samples": model.normalizer.inverse_future(x_noisy)})

samples = DiffusionProcessVisualizer.batch_to_samples(samples)
DiffusionProcessVisualizer.plot_diffusion(samples[0])

In [ ]:

# run_validate(il_cfg)
# run_test(il_cfg)


result = run_predict(il_cfg)


In [ ]:
# result[0]["batch"]["future"].shape

batch_index = 5
batch = result[batch_index]["batch"]
batch["pred_future"]=result[batch_index]["pred_future"]
samples = TrajectoryDatasetVisualizer.batch_to_samples(batch)
print(len(batch)), print(len(samples))
TrajectoryDatasetVisualizer.plot(samples)

## 检查 batch 结构

In [ ]:
# batch = next(iter(train_loader))
print("Batch keys 和 shape:")
for key, val in batch.items():
    print(f"  {key:25s}: {val.shape}  dtype={val.dtype}")

print(f"\nvehicle_params 示例 (第一个样本):")
print(f"  [a_max, omega_max, v_max, wheelbase, length, width, height,")
print(f"   front_overhang, rear_overhang, mass, drag_coeff]")
print(f"  {batch['vehicle_params'][0].cpu().numpy()}")